In [ ]:
import json
import pandas as pd
import os
from pathlib import Path

# Run this notebook from within the 09_llm_categorization/ directory.
# All intermediate JSONL files are saved here; input/output CSVs use relative paths.
notebook_dir = Path.cwd()
project_root = notebook_dir.parent

data = pd.read_csv(project_root / "data" / "raw" / "literature_dataset.csv")
dataShort = data

In [ ]:
# chunk not needed if you already have a parsed column

def parse_keywords(row):
    # Get Title and Abstract, treating NaN as an empty string
    parsed_input = (row['Title'] if pd.notna(row['Title']) else '') + " " + (row['Abstract'] if pd.notna(row['Abstract']) else '')
    
    # Check for keywords in the specified order and add the first non-empty one found
    if pd.notna(row['Author Keywords']) and row['Author Keywords'].strip():
        parsed_input += " " + row['Author Keywords']
    elif pd.notna(row['WOS Keywords']) and row['WOS Keywords'].strip():
        parsed_input += " " + row['WOS Keywords']
    elif pd.notna(row['Scopus Keywords']) and row['Scopus Keywords'].strip():
        parsed_input += " " + row['Scopus Keywords']

    return parsed_input

# Apply the function row by row
# dataShort['parsedInput'] = dataShort.apply(parse_keywords, axis=1)

# def parse_keywords(row):
#     # Get Title and Abstract, treating NaN as an empty string
#     parsed_input = (row['Title'] if pd.notna(row['Title']) else '') + " " + (row['Abstract'] if pd.notna(row['Abstract']) else '')
    
#     # Check for keywords in the specified order and add the first non-empty one found
#     if pd.notna(row['Author Keywords']) and row['Author Keywords'].strip():
#         parsed_input += " " + row['Author Keywords']
#     elif pd.notna(row['Scopus Keywords']) and row['Scopus Keywords'].strip():
#         parsed_input += " " + row['Scopus Keywords']
#     elif pd.notna(row['WOS Keywords']) and row['WOS Keywords'].strip():
#         parsed_input += " " + row['WOS Keywords']
    
#     return parsed_input

# Apply the function row by row
dataShort['parsedInput'] = dataShort.apply(parse_keywords, axis=1)

In [ ]:
# List to hold all API call objects
batch_requests = []
for idx, parsedText in enumerate(dataShort['parsedInput'], start=1):
    batch_requests.append({
        "custom_id": f"{dataShort['ID'][idx-1]}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            # "model": "gpt-5-mini", # choose one of the two
            "model": "o4-mini", # choose one of the two
            "reasoning_effort": "medium", # that's default
            "n": 5, # maybe go up to 5 for the final run?
            "temperature": 1, # would do smaller for non-reasoning models, but reasoning models are 1 by default
            "messages": 
                [{"role": "developer", "content": "You are an expert at reviewing papers in the social science related to air pollution."},
                 {"role": "user", "content": f"""
             This is the title, abstract, and keywords for a given research paper: "```{parsedText}```"
             This paper uses survey data on air pollution, please classify this paper as being in one or multiple of the following categories:
             
             -Perceptions: This category focuses on studies that explore how people perceive the issue of air pollution. 
             This includes awareness of air pollution, perceived risks, perceptions of personal or community exposure, concern and knowledge about pollution, and beliefs about its sources.
             The key differentiating factor is that this category captures what people think, know or believe about air pollution, rather than what they do or are willing to do.

             -Behaviors: This category focuses on studies that examine what people report they already do, or are willing to do, in response to air pollution.
             This could include actual (revealed) behaviors, stated behaviors, or expressed willingness to act.
             The behavior must relate specifically to air pollution and may include protective actions, lifestyle changes, or other forms of individual or collective response.
             The key distinction is that the study focuses on concrete or intended actions taken in relation to air pollution.
             The results should be the focus of the study, while any text introducing behavior with air pollution as an example can be ignored.

             -Policy: This category includes studies that examine public attitudes or preferences toward policies related to air pollution.
             This may include preferences for pollution reduction strategies, support for policies that facilitate adaptation, opinions about what the government should do, perceived policy effectiveness, or willingness to pay for policies.
             The key criterion is that the study includes an evaluation or stated preference regarding some form of government or institutional action (either current, past or proposed) that directly addresses air pollution.

             -Health: This category includes studies that identify health-related impacts, perceptions, or behaviors associated with air pollution, using survey data.
             This could include self-reported health indicators, perceptions of health risks, or beliefs about how pollution affects personal or community health.
             The defining feature is a clear connection between air pollution and a health-related impact, behavior, or perception within the setting of a survey.

             -Priority: This category focuses on studies that assess how important people consider air pollution, either in absolute terms or relative to other social, economic, or environmental issues.
             This may include stated concern levels, issue rankings, or perceptions of urgency regarding air quality.
             The key distinguishing feature is that the study focuses on the perceived priority of air pollution in the minds of respondents.

             The paper can cover one or multiple categories. If it does not cover any of these categories, please return None as a category.
             
             Steps to follow:
             - write a small explanation of why you would categorize a paper in one or multiple of the categories above. 
             - write "output_json".
             - Then make a new line and only write, in a JSON format between curly brackets, the output where the key is "categories" and the value is a list of categories that the paper is covering such as:
                {{
                    "categories": ["Category1", "Category2", ...]
                }}
             """
             }],
            # "max_tokens": 300 # if using gpt-4.1-mini or similar
            'max_completion_tokens': 1000, # if using o4-mini or similar reasoning model
        }
    })

with open("batch_requests_classification_o4mini_final.jsonl", "w", encoding="utf-8") as fout:
    for req in batch_requests:
        fout.write(json.dumps(req) + "\n")

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="INSERT-YOUR-KEY-HERE")

In [ ]:
batch_input_file = client.files.create(
    file=open("batch_requests_classification_o4mini_final.jsonl", "rb"),
    purpose="batch"
)

print(batch_input_file)

In [ ]:
batch_input_file_id = batch_input_file.id
batch_created = client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "description": "classification papers o4mini n5 fullCall",
    }
)

print(batch_created)
print(batch_created.id)
batch_id = batch_created.id

In [ ]:
# 2) Retrieve the batch by its ID
# batch = client.batches.retrieve('batch_68b00b4905008190ad080a7e91c29992')
batch = client.batches.retrieve(batch_id)

# 3) Print the full Batch object
print(batch)

# extract the batch ID and output file id from the response
batch_id = batch.id
output_file_id = batch.output_file_id

error_file_id = batch.error_file_id

In [ ]:
# check for error
file_error = client.files.content(error_file_id) # latest batch_68aebdcae2e88190ad8146179029f72d
file_error.text

In [ ]:
file_response = client.files.content(output_file_id) # latest batch_6818ed41d9848190989112e6ec43fbff
print(file_response.text)

file_response.write_to_file("batch_requests_classification_o4mini_final_response.jsonl")

In [ ]:
# import json
# import pandas as pd

# 1. Read your JSONL
with open("batch_requests_classification_o4mini_final_response.jsonl", "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

# 2. Normalize, exploding the choices list into rows
df = pd.json_normalize(
    records,
    record_path=["response", "body", "choices"],
    meta=[
        "custom_id",
        ["response", "body", "model"],
        ["response", "body", "created"],
        ["id"]
    ],
    errors="ignore"
)

# 4. Clean up column names if you like
df = df.rename(
    columns={
        "index": "choice_index",
        "message.content": "message_content"
    }
)

print(df)

In [ ]:
import re

def extract_identification(text):
    # Use regular expressions to find the JSON portion of the text
    if not text:  # Check if text is empty or None
        return None
    json_match = re.search(r'\{.*?\}', text, re.DOTALL)
    
    if json_match:
        # Extract the JSON string
        json_str = json_match.group(0)
        
        # Load the JSON string into a Python dictionary
        json_data = json.loads(json_str)
        
        # Extract the value of the 'countries' key
        classification = json_data.get("categories", None)
        return str(classification)
    else:
        return None

df['classificationChatGPT'] = df['message_content'].apply(extract_identification)
df['classificationChatGPTClean'] = df['classificationChatGPT'].str.replace(r"[\[\]'\"]", "", regex=True)


In [ ]:
# select important columns and save to csv
df_save = df[['custom_id', "choice_index", 'message_content', 'classificationChatGPTClean', 'response.body.model']]

df_save.to_csv(project_root / "data" / "llm_classifications" / "classification_o4mini.csv", index=False)